# LLM implementation from Scratch

This notebook aims at implementing all the different parts necessary to make an LLM work, following the Sebastian Raschka guide.

## Data Preparation and Tokenization


### Basic Tokenizers

In [1]:
#dataset load
import os
import urllib.request

if not os.path.exists('/data/the-verdict.txt'):
    url = ('https://raw.githubusercontent.com/rasbt/'
           'LLMs-from-scratch/main/ch02/01_main-chapter-code/'
           'the-verdict.txt')
    file_path = 'the-verdict.txt'
    urllib.request.urlretrieve(url, file_path)

In [2]:
with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()
len(raw_text)

20479

In [3]:
import re

text = 'Hello, world! My name is Riccardo and this is a test.'
result = re.split(r'([,.;:!?_"()\']|--|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '!', '', ' ', 'My', ' ', 'name', ' ', 'is', ' ', 'Riccardo', ' ', 'and', ' ', 'this', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [4]:
result = [item for item in result if item.strip()] # remove empty strings
print(result)

['Hello', ',', 'world', '!', 'My', 'name', 'is', 'Riccardo', 'and', 'this', 'is', 'a', 'test', '.']


In [5]:
result_rt = re.split(r'([,.;:!?_"()\']|--|\s)', raw_text)
result_rt = [item.strip() for item in result_rt if item.strip()]
preprocessed = result_rt
len(preprocessed)

4690

In [6]:
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

In [7]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

In [8]:
vocab = {token: integer for integer, token in enumerate(all_words)}
vocab

{'!': 0,
 '"': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '--': 6,
 '.': 7,
 ':': 8,
 ';': 9,
 '?': 10,
 'A': 11,
 'Ah': 12,
 'Among': 13,
 'And': 14,
 'Are': 15,
 'Arrt': 16,
 'As': 17,
 'At': 18,
 'Be': 19,
 'Begin': 20,
 'Burlington': 21,
 'But': 22,
 'By': 23,
 'Carlo': 24,
 'Chicago': 25,
 'Claude': 26,
 'Come': 27,
 'Croft': 28,
 'Destroyed': 29,
 'Devonshire': 30,
 'Don': 31,
 'Dubarry': 32,
 'Emperors': 33,
 'Florence': 34,
 'For': 35,
 'Gallery': 36,
 'Gideon': 37,
 'Gisburn': 38,
 'Gisburns': 39,
 'Grafton': 40,
 'Greek': 41,
 'Grindle': 42,
 'Grindles': 43,
 'HAD': 44,
 'Had': 45,
 'Hang': 46,
 'Has': 47,
 'He': 48,
 'Her': 49,
 'Hermia': 50,
 'His': 51,
 'How': 52,
 'I': 53,
 'If': 54,
 'In': 55,
 'It': 56,
 'Jack': 57,
 'Jove': 58,
 'Just': 59,
 'Lord': 60,
 'Made': 61,
 'Miss': 62,
 'Money': 63,
 'Monte': 64,
 'Moon-dancers': 65,
 'Mr': 66,
 'Mrs': 67,
 'My': 68,
 'Never': 69,
 'No': 70,
 'Now': 71,
 'Nutley': 72,
 'Of': 73,
 'Oh': 74,
 'On': 75,
 'Once': 76,
 'Only': 77,
 '

In [9]:
class BasicTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    

    def encode(self, text):
        preprocessed = re.split(r'([,.;:!?_"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        #replace spaces before the punctuation
        text = re.sub(r'\s+([,.?!"\'])', r'\1', text)
        return text

In [10]:
tokenizerv1 = BasicTokenizerV1(vocab)
sample_text = """It's the last he painted, you know,
Mrs. Gisburn said with pardonable pride."""
print(tokenizerv1.encode(sample_text))
print(tokenizerv1.decode(tokenizerv1.encode(sample_text)))

[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 67, 7, 38, 851, 1108, 754, 793, 7]
It' s the last he painted, you know, Mrs. Gisburn said with pardonable pride.


In [11]:
#handle unknown words
all_words = sorted(set(preprocessed))
all_words.extend(['<|unk|>', '<|endoftext|>'])
vocab = {token:integer for integer, token in enumerate(all_words)}

class BasicTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    

    def encode(self, text):
        preprocessed = re.split(r'([,.;:!?_"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        #make it unk
        preprocessed = [item if item in self.str_to_int
                        else '<|unk|>' for item in preprocessed]
        
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        #replace spaces before the punctuation
        text = re.sub(r'\s+([,.?!"\'])', r'\1', text)
        return text
    

In [12]:
tokenizerV2 = BasicTokenizerV2(vocab)

print(tokenizerV2.encode("Hello, my name is Riccardo!"))
print(tokenizerV2.decode(tokenizerV2.encode("Hello, my name is Riccardo!")))

[1130, 5, 697, 1130, 584, 1130, 0]
<|unk|>, my <|unk|> is <|unk|>!


### Byte Pair encoding

- implementation is quite complex and is skipped for the moment, gonna use tiktoken instead (rust based library created by openai)
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/05_bpe-from-scratch/bpe-from-scratch-simple.ipynb seb implementation


In [13]:
import tiktoken
tiktoken.__version__

'0.12.0'

In [14]:
tokenizer = tiktoken.get_encoding("gpt2")
data_tokens = tokenizer.encode(raw_text)
data_tokens[:10]  # first 10 tokens

[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]

### Data Sampling with sliding window

In [15]:
enc_sample = data_tokens[50:]

In [16]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1: context_size+1]

for i in range(1, context_size+1):
    context = enc_sample[:i]
    target = enc_sample[i]

    print(tokenizer.decode(context), '---->', tokenizer.decode([target]))


 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
torch.__version__

'2.9.1+cu128'

In [18]:
class GPTDatasetV1(Dataset):
    
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        #tokenize
        token_ids = tokenizer.encode(txt, allowed_special = {"<|endoftext|>"})

        #use a sliding window to chunk the book into overlapping sequences
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i+1 : i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx] 

Param explanation for _create_dataloader_v1_:
- drop_last is to avoid unstable training if the dataset is not divisible bby the batch size precisely, therefore creating a last batch which is not of the same size of the others
- num_workers is referring to the number of background processes used at the same time (can cause problems if not set to 0 on notebooks)

In [19]:
def create_dataloader_v1(txt, batch_size = 4, max_lenght = 256,
                         stride = 128, shuffle = True, drop_last = True,
                         num_workers = 0):
    
    #Initialize tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    #Create Dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_lenght, stride)

    #Create DataLoader
    dataloader = DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = shuffle,
        drop_last = drop_last,
        num_workers = num_workers
    )

    return dataloader

In [20]:
with open('the-verdict.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

In [21]:
dataloader = create_dataloader_v1(txt= raw_text,
                                batch_size=8,
                                max_lenght=4,
                                stride=4, #every example becomes unique
                                shuffle=False,
                                )

data_iter = iter(dataloader)

In [22]:
inputs, targets = next(data_iter)
print("Inputs:", inputs)
print("Targets:", targets)

Inputs: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Creating Embeddings

We will be using the _torch.nn.embedding_ 'cause it's a more efficient implementation of a linear layer specified for embeddings. 

#### Under the hood implementation (uh)

We basically want 
- $\Omega{e}$ which has dimension (Dmodel, Vocab)
- _T_ which has dimension (Vocab, N_seq)

So that we can create $X = \Omega{e}T$ which is (Dmodel, N_seq) (in the implementation we actually get (N_seq,Dmodel) tho)

Therefore _T = onehot_uh.T_ and $\Omega{e}$ = _embedding_uh.weight.T_


In [23]:
uh_ids = torch.tensor([2,3,1])
onehot_uh = torch.nn.functional.one_hot(uh_ids)
num_idx = max(uh_ids) + 1
n_dim = 6
print("One-hot encoded:\n", onehot_uh)

One-hot encoded:
 tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])


In [24]:
# Let also create it with nn.embeddings

torch.manual_seed(42)
embedding_uh = torch.nn.Embedding(num_embeddings=num_idx,
                                    embedding_dim=n_dim)
print(embedding_uh.weight) # which is a matrix of shape (vocab, d)
print(embedding_uh)

Parameter containing:
tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345],
        [-0.0431, -1.6047,  0.3559, -0.6866, -0.4934,  0.2415],
        [-1.1109,  0.0915, -2.3169, -0.2168, -0.3097, -0.3957],
        [ 0.8034, -0.6216, -0.5920, -0.0631, -0.8286,  0.3309]],
       requires_grad=True)
Embedding(4, 6)


In [25]:
#seed for reproducibility
linear_uh = torch.nn.Linear(in_features=num_idx,
                            out_features= n_dim,
                            bias=False)
print(linear_uh)

linear_uh.weight = torch.nn.Parameter(embedding_uh.weight.T)
linear_uh.weight

Linear(in_features=4, out_features=6, bias=False)


Parameter containing:
tensor([[ 1.9269, -0.0431, -1.1109,  0.8034],
        [ 1.4873, -1.6047,  0.0915, -0.6216],
        [ 0.9007,  0.3559, -2.3169, -0.5920],
        [-2.1055, -0.6866, -0.2168, -0.0631],
        [ 0.6784, -0.4934, -0.3097, -0.8286],
        [-1.2345,  0.2415, -0.3957,  0.3309]], requires_grad=True)

In [26]:
linear_uh(onehot_uh.float())

tensor([[-1.1109,  0.0915, -2.3169, -0.2168, -0.3097, -0.3957],
        [ 0.8034, -0.6216, -0.5920, -0.0631, -0.8286,  0.3309],
        [-0.0431, -1.6047,  0.3559, -0.6866, -0.4934,  0.2415]],
       grad_fn=<MmBackward0>)

In [27]:
embedding_uh(uh_ids)

tensor([[-1.1109,  0.0915, -2.3169, -0.2168, -0.3097, -0.3957],
        [ 0.8034, -0.6216, -0.5920, -0.0631, -0.8286,  0.3309],
        [-0.0431, -1.6047,  0.3559, -0.6866, -0.4934,  0.2415]],
       grad_fn=<EmbeddingBackward0>)

### Back to the main example

In [28]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(num_embeddings = vocab_size,
                                           embedding_dim = output_dim)

In [42]:
max_lenght = 4
batch_size = 8
stride = 4
shuffle = False
dataloader = create_dataloader_v1(raw_text,
                                batch_size,
                                max_lenght,
                                stride, #every example becomes unique
                                shuffle,
                                )

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
inputs.shape  # (batch_size, seq_length)

torch.Size([8, 4])

In [30]:
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape #(Bathch_size, Seq_length, Embedding_dim)

torch.Size([8, 4, 256])

### Adding positional infos

In this case pos embeddings are not fixed like in the Vaswani paper but instead are learned within the LLM.

In [31]:
context_length = max_lenght

pos_embedding_layer = torch.nn.Embedding(num_embeddings = context_length,
                                        embedding_dim = output_dim)

pos_embedding = pos_embedding_layer(torch.arange(context_length))
print(f"Positional embeddings shape: {pos_embedding.shape}")


#added position embeddings to token embeddings
input_embeddings = token_embeddings + pos_embedding
print(f"Input embeddings shape: {input_embeddings.shape}") #broadcasting is applied

Positional embeddings shape: torch.Size([4, 256])
Input embeddings shape: torch.Size([8, 4, 256])


## Coding Attention Mechanism

torch.Parameter is a wrapper around a tensor to make it trainable

In [49]:
d_model = 256
h = 1
d_k = d_model/h

In [85]:
import torch.nn as nn

class SelfAttention(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False, causal=True):

        super().__init__()
        self.W_q = torch.nn.Linear(output_dim, output_dim, bias=qkv_bias)
        self.W_k = torch.nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_v = torch.nn.Linear(d_in, d_out, bias = qkv_bias)
        self.causal = causal

    def forward(self, x):
        queries = self.W_q(x)
        keys = self.W_k(x)
        values = self.W_v(x)

        attn_scores = queries @ keys.transpose(-2, -1)

        if self.causal:
            mask = torch.triu(torch.ones(attn_scores.size(-2), attn_scores.size(-1)), diagonal = 1)
            attn_scores = attn_scores.masked_fill(mask == 1, float('-inf'))

        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim = -1)
        context_vector = attn_weights @ values

        return context_vector

torch.manual_seed(42)
sa = SelfAttention(d_model, d_model)
sa(input_embeddings)


tensor([[[-1.5567, -1.1865, -0.6202,  ...,  0.0511, -0.3291, -0.0467],
         [-1.0190, -0.6398, -0.2807,  ...,  0.2533, -0.2419,  0.1322],
         [-0.4859, -0.7074, -0.0825,  ...,  0.1743, -0.5493,  0.0585],
         [-0.5788, -0.5966,  0.0369,  ...,  0.1933, -0.4373,  0.1035]],

        [[-0.0913, -1.2160,  0.5934,  ...,  0.5537,  0.3587,  0.3375],
         [-0.8010, -0.4511,  0.7630,  ...,  0.7686,  0.2820, -0.1627],
         [-0.2233, -0.0022,  0.6412,  ...,  0.9412, -0.0785,  0.1668],
         [-0.4003, -0.0055,  0.7096,  ...,  0.8114,  0.0563, -0.0980]],

        [[-1.9010, -1.4015, -0.4580,  ..., -0.4413, -0.2390, -0.4186],
         [-1.4736, -0.9148, -0.3258,  ...,  0.0255, -0.0067, -0.5553],
         [-0.1989, -1.0618,  0.2519,  ...,  0.8990, -1.2517, -0.8745],
         [-1.0994, -1.2430, -0.1063,  ...,  0.0335, -0.6440, -0.6428]],

        ...,

        [[-0.5200, -0.9054, -1.0429,  ...,  0.5762, -0.3713,  0.7520],
         [-1.3278, -0.1260, -0.1525,  ...,  0.2644, -0.42

In [ ]:
import torch.nn as nn

d_model = 256
h = 4
d_k = d_model/h



class MHAttention(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False, causal=True):

        super().__init__()
        self.W_q = torch.nn.Linear(output_dim, output_dim, bias=qkv_bias)
        self.W_k = torch.nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_v = torch.nn.Linear(d_in, d_out, bias = qkv_bias)
        self.causal = causal

    def forward(self, x):
        queries = self.W_q(x)
        keys = self.W_k(x)
        values = self.W_v(x)

        attn_scores = queries @ keys.transpose(-2, -1)

        if self.causal:
            mask = torch.triu(torch.ones(attn_scores.size(-2), attn_scores.size(-1)), diagonal = 1)
            attn_scores = attn_scores.masked_fill(mask == 1, float('-inf'))

        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim = -1)
        context_vector = attn_weights @ values

        return context_vector
